In [44]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error

# START: Vivian Lin

### Load Dataset

Feature Selection:
- `LivingArea`
- `BathroomsTotalIntege`
- `BedroomsTotal`
- `Bed_Bath_Ratio`
- `Property_Age`
- `Months_From_Dec_2025`
- `Sqft_Per_Bedroom`
- `Lot_Utilization`

Dependent Variable: `LogClosePrice`

In [24]:
tree_features = [
    'LivingArea','BathroomsTotalInteger',
    'BedroomsTotal','Bed_Bath_Ratio','Property_Age',
    'Months_From_Dec_2025','Sqft_Per_Bedroom','Lot_Utilization'
]

`X_train`/`X_test`: feature metrics

`y_train`/`y_test`: original target

In [56]:
# Load dataset with feature engineering
train = pd.read_csv('../processed_data/train_cleaned_fe.csv', engine='python', on_bad_lines='skip')
test  = pd.read_csv('../processed_Data/test_cleaned_fe.csv', engine='python', on_bad_lines='skip')

# Replace inf and -inf with NAN
train = train.replace([np.inf, -np.inf], np.nan)

train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice'] = np.log(test['ClosePrice'])

#Drop rows with NAN values
train = train.dropna()
test = test.dropna()

# X and y of train set
X_train = train[tree_features]
y_train = train['LogClosePrice']

# X and y of test set
X_test = test[tree_features]
y_test = test['LogClosePrice']

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

#### Fitting without hyperparameter tuning

In [39]:
xg_model = XGBRegressor()
xg_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

#### Evaluating XGBoost Model with Default Hyperparameters

In [60]:
# Model Predictions
no_hyper_pred_train = xg_model.predict(X_train)
no_hyper_pred_test = xg_model.predict(X_test)

# R^2 on a Log Scale
no_hyper_R2_train = r2_score(no_hyper_pred_train, y_train)
print(f'Train Set -- R^2 on Log Scale: {no_hyper_R2_train}')

no_hyper_R2_test = r2_score(no_hyper_pred_test, y_test)
print(f'Test Set -- R^2 on Log Scale: {no_hyper_R2_test}')

# MAPE
no_hyper_mape_train = mean_absolute_percentage_error(no_hyper_pred_train, y_train)
print(f'Train Set -- MAPE: {no_hyper_mape_train*100}%')

no_hyper_mape_test = mean_absolute_percentage_error(no_hyper_pred_test, y_test)
print(f'Test Set -- MAPE: {no_hyper_mape_test*100}%')

#MdAPE
no_hyper_mdape_train = np.median(np.abs((y_train - no_hyper_pred_train) / y_train))
print(f'Train Set -- MdAPE: {no_hyper_mdape_train * 100}%')

no_hyper_mdape_test = np.median(np.abs((y_test - no_hyper_pred_test) / y_test))
print(f'Test Set -- MdAPE: {no_hyper_mdape_test * 100}%')


Train Set -- R^2 on Log Scale: 0.3136224994076525
Test Set -- R^2 on Log Scale: 0.19007401498888432
Train Set -- MAPE: 2.1486251657346944%
Test Set -- MAPE: 2.329129519335367%
Train Set -- MdAPE: 1.692807949882932%
Test Set -- MdAPE: 1.8539303424727518%


### Key Evaluation Metrics

| Metric       | Training Set | Testing Set |
|-------------|-------------|------------|
| **R²**      | 0.314      | 0.190      |
| **MAPE (%)**| 2.149%       | 2.329%      |
| **MdAPE (%)**| 1.692%      | 1.854%      |

### Interpretation

1. **Low R²**: The model explains ~31% of variance on training data and ~19% on unseen test data, indicating that most of the variation in the target variable cannot be explained by the model.  
2. **Extremely Low MAPE and MdAPE**: The mean and median absolute percentage errors are extremely low, meaning predictions are, on average, within ~1-2% of the true property prices.
3. **Robustness**: The small difference between training and testing MAPE and MdAPE suggests the model is not overfitting and handles unseen data well. However, the large difference between the R^2 indicates that the model explains much less variance in the test data than in the training data, suggesting that the model may not capture all of the underlying patterns in the data or that important predictors may be missing.
4. **Practical Use**: This XGBoost model is effective at producing accurate predictions of residential property closing prices, as indicated by the low MAPE and MdAPE values. However, the relatively low $R^2$ suggests that the model explains only a limited portion of the variation in the target variable LogClosePrice, meaning that important factors influencing property prices may not be captured by the current feature set.

# END: Vivian Lin